# 05 — Task-Related Component Analysis (TRCA)

**Owner:** Ishanvir

**Reference:** Nakanishi, M., Wang, Y., Chen, X., Wang, Y.-T., Gao, X., & Jung, T.-P. (2018). *Enhancing Detection of SSVEPs for a High-Speed Brain Speller Using Task-Related Component Analysis.* IEEE Trans. Biomed. Eng. 65(1):104–112.

**This notebook demonstrates a documented negative result.** TRCA underperforms CCA and FBCCA on this dataset by 30–40 percentage points at every fold under every protocol. The implementation is correct (verified at 9e-15 precision against the meegkit reference — see `results/audit_report.md` Layer 10.3). 

The negative result is a property of the data, not the code.

In [1]:
import os
from pathlib import Path
_here = Path.cwd()
_root = next((p for p in [_here, *_here.parents] if (p / 'data' / 'raw').exists()), None)
if _root and Path.cwd() != _root:
    os.chdir(_root)

import numpy as np
import scipy.linalg as linalg

from ssvep.io import load_all
from ssvep.classifiers import TRCAClassifier
from ssvep.classifiers.trca import _trca
from ssvep.evaluation import leave_one_block_out_cv, itr


## 1. Load data


In [2]:
d = load_all(window_s=3.0)
X, y, blocks = d['X'], d['y'], d['blocks']
fs, sf = d['fs'], d['stim_freqs']
print(f"X.shape: {X.shape}; fs={fs}; stim_freqs={sf}")


X.shape: (80, 8, 768); fs=256.0; stim_freqs=[ 9. 10. 12. 15.]


## 2. The eigenvalue solve (one class, one training block)

For each class, TRCA solves the generalized eigenvalue problem `S w = λ Q w`:

- **Q**: within-class scatter on concatenated, per-channel mean-centered trials. `Q = UX · UX^T` where `UX` is the concatenated `(n_chans, n_samples · n_trials)` matrix.
- **S**: pairwise inter-trial scatter, summed over distinct trial pairs:
  `S = Σ_{i<j} (x_i^T · x_j + x_j^T · x_i)` after each trial is mean-centered along the samples axis.

The eigenvector with the largest eigenvalue is the spatial filter that maximizes inter-trial reproducibility within the class.


In [3]:
# Pick training block 0, class 0 (5 trials × 8 channels × 768 samples)
train_mask = blocks == 0
Xc = X[train_mask][y[train_mask] == 0]                 # (5, 8, 768)
n_trials, n_chans, n_samples = Xc.shape
print(f"class-0 training trials: {Xc.shape}")

# Internal convention: (n_samples, n_chans, n_trials)
X_internal = Xc.transpose(2, 1, 0)                     # (768, 8, 5)

# Build Q
UX = np.zeros((n_chans, n_samples * n_trials))
for trial in range(n_trials):
    UX[:, trial*n_samples:(trial+1)*n_samples] = X_internal[..., trial].T
UX -= np.mean(UX, axis=1, keepdims=True)
Q = UX @ UX.T

# Build S
S = np.zeros((n_chans, n_chans))
for i in range(n_trials - 1):
    x1 = X_internal[..., i].copy(); x1 -= np.mean(x1, axis=0)
    for j in range(i + 1, n_trials):
        x2 = X_internal[..., j].copy(); x2 -= np.mean(x2, axis=0)
        S += x1.T @ x2 + x2.T @ x1

lambdas, W = linalg.eig(S, Q, left=True, right=False)
order = np.argsort(-np.real(lambdas))
print(f"\neigenvalues (sorted desc): {np.real(lambdas[order]).round(3)}")
filt = np.real(W[:, order[0]])
print(f"top spatial filter        : {filt.round(4)}")
print(f"||filter||_2              : {np.linalg.norm(filt):.6f}    (expect 1.0 from generalized eig)")

# Cross-check against the project _trca
proj_filt = _trca(X_internal)
diff_a = np.max(np.abs(proj_filt - filt))
diff_b = np.max(np.abs(proj_filt + filt))
print(f"\nmax-abs-diff vs project _trca (allowing sign flip): {min(diff_a, diff_b):.2e}")


class-0 training trials: (5, 8, 768)

eigenvalues (sorted desc): [ 0.492  0.231  0.097  0.05  -0.001 -0.091 -0.114 -0.235]
top spatial filter        : [-0.0471  0.0371  0.1918  0.3585 -0.1338 -0.2403  0.4335 -0.7534]
||filter||_2              : 1.000000    (expect 1.0 from generalized eig)

max-abs-diff vs project _trca (allowing sign flip): 9.02e-15


## 3. Ensemble TRCA: stack per-class filters into `W_ensemble`

Repeat the eigenvalue solve for each class, then stack the per-class filters into `W ∈ R^{n_chans × n_classes}`. At predict time, project both the test trial and each per-class template through `W_ensemble`, flatten, and correlate.


In [4]:
clf = TRCAClassifier(stim_freqs=sf, fs=fs, ensemble=True)
train_mask = blocks == 0
clf.fit(X[train_mask], y[train_mask])
print(f"Per-class filters: {sorted(clf.spatial_filters_.keys())}")
print(f"W_ensemble shape : {clf.W_ensemble_.shape}    (expect (8, 4))")
print(f"\nW_ensemble:\n{clf.W_ensemble_.round(3)}")


Per-class filters: [0, 1, 2, 3]
W_ensemble shape : (8, 4)    (expect (8, 4))

W_ensemble:
[[-0.047 -0.014 -0.004 -0.368]
 [ 0.037 -0.15   0.44   0.6  ]
 [ 0.192 -0.029  0.107 -0.601]
 [ 0.359 -0.083 -0.052 -0.127]
 [-0.134 -0.23   0.459  0.867]
 [-0.24  -0.174 -0.169  0.782]
 [ 0.433 -0.182 -0.722 -1.   ]
 [-0.753  0.924 -0.181 -0.213]]


## 4. Cross-subject 4-block LOBO

The standard protocol used by `compare_classifiers.py` (no `--protocol` flag = cross-subject default). Each fold trains on 3 blocks (mixed-subject) and tests on the held-out one.


In [5]:
factory = lambda: TRCAClassifier(stim_freqs=sf, fs=fs, ensemble=True)
result_xs = leave_one_block_out_cv(X, y, blocks, factory)
print(f"per-block accuracies: {result_xs['per_block']}")
print(f"mean ± std         : {result_xs['mean']:.3f} ± {result_xs['std']:.3f}")
print(f"ITR @ 3.0s window  : {itr(result_xs['mean'], n_classes=4, window_s=3.0):.2f} bits/min")


per-block accuracies: {0: 0.25, 1: 0.3, 2: 0.35, 3: 0.35}
mean ± std         : 0.312 ± 0.041
ITR @ 3.0s window  : 0.29 bits/min


## 5. Within-subject 4-fold LOBO (Nakanishi 2018 protocol)

Train on one block, test on the other block from the same subject (4 folds: 2 per subject). This is the protocol Nakanishi et al. report against, it gives TRCA the fairest possible shot on this dataset.


In [6]:
WITHIN_SUBJECT_FOLDS = [(0, 1), (1, 0), (2, 3), (3, 2)]   # (train, test); subj 1 first, subj 2 last

per_block_w = {}
for train_b, test_b in WITHIN_SUBJECT_FOLDS:
    train_mask = blocks == train_b
    test_mask = blocks == test_b
    clf = TRCAClassifier(stim_freqs=sf, fs=fs, ensemble=True)
    clf.fit(X[train_mask], y[train_mask])
    per_block_w[int(test_b)] = float(clf.score(X[test_mask], y[test_mask]))

accs = np.array(list(per_block_w.values()))
mean_w = float(accs.mean()); std_w = float(accs.std())
# Ordering of per_block_w keys preserves insertion (test blocks 1, 0, 3, 2 → subj 1 first 2, subj 2 last 2)
accs_in_fold_order = list(per_block_w.values())
subj1_mean = float(np.mean(accs_in_fold_order[:2]))
subj2_mean = float(np.mean(accs_in_fold_order[2:]))

print(f"per-test-block accuracies (fold order): {per_block_w}")
print(f"mean ± std        : {mean_w:.3f} ± {std_w:.3f}")
print(f"subject 1 mean    : {subj1_mean:.3f}    (expect ≈ 0.675)")
print(f"subject 2 mean    : {subj2_mean:.3f}    (expect ≈ 0.275)")
print(f"ITR @ 3.0s window : {itr(mean_w, n_classes=4, window_s=3.0):.2f} bits/min")


per-test-block accuracies (fold order): {1: 0.7, 0: 0.65, 3: 0.3, 2: 0.25}
mean ± std        : 0.475 ± 0.202
subject 1 mean    : 0.675    (expect ≈ 0.675)
subject 2 mean    : 0.275    (expect ≈ 0.275)
ITR @ 3.0s window : 3.39 bits/min


## 6. Why TRCA underperforms

The full analysis is in `results/trca_methodology_note.md`. The summary:

### (a) Training data scarcity vs Nakanishi's 11-trial saturation threshold

Nakanishi et al. (2018) report that ensemble TRCA's lift over plain CCA materializes around 5 trials per class and **saturates** near 11. Within-subject LOBO on this dataset has **exactly 5**, the absolute floor of the data where any TRCA benefit appears at all. Below that floor, the eigenvalue solver's S/Q matrices don't have enough pairwise scatter to identify a stable maximum-reproducibility direction; the spatial filter is dominated by sample noise.

### (b) CCA's matched-filter optimality on widely spaced stim frequencies

Our 4 stim frequencies (9, 10, 12, 15 Hz) are spaced 1–3 Hz apart with no harmonic overlap up to the 5th harmonic. For this kind of stimulus design, CCA's canonical sin/cos references are a near-orthogonal basis. There is no slack for an "advanced" classifier to exploit.

The data where TRCA outperforms CCA in published work (Nakanishi 2018, Chen 2015) is the **opposite**: 40-target spellers with 0.2 Hz spacing, where CCA's references heavily overlap and matched-filter optimality breaks down. That is not this dataset.

### (c) Subject 2's inter-session inconsistency

Subject 2's TRCA within-subject mean is 0.275, within rounding of the 0.250 chance level. This is the strongest possible statement: the SSVEP response itself is not stable across subject 2's two sessions. A spatial filter learned on block 2 doesn't generalize to block 3, and vice versa. TRCA, which by construction relies on inter-trial reproducibility, has nothing to latch onto. CCA's external sin/cos reference works because it doesn't require inter-session consistency in the first place.

### Cross-subject is *actively harmful* (not just uninformative)

Compare subject 1's per-block accuracy under within-subject vs cross-subject:

| Block | Subject | Within-subject (train = same subject's other block) | Cross-subject (train = the other 3 blocks, including subject 2) |
|---|---|---|---|
| 0 | 1 | 0.65 | 0.25 |
| 1 | 1 | 0.70 | 0.30 |

Adding subject 2's data to subject 1's training set drops subject 1 from ≈0.675 to ≈0.275. The ensemble spatial filter learned from a mixed-subject training set is dominated by whichever subject contributes higher-variance signal, and applies a transformation that destroys both subjects' class structure.

### Conclusion

TRCA is implemented correctly (the negative result is reproducible; the eigenvalue solve matches meegkit to machine precision) and the result is consistent with Nakanishi 2018's own published thresholds. This dataset sits below those thresholds on multiple axes simultaneously. The negative result is a documented refusal to generalize where the data does not support it. This is not a bug.

For the comparable positive case, see notebook 04: FBCCA, which does not need training trials at all, recovers harmonic energy that CCA misses, and lifts subject 2 without losing subject 1.